# Eiger4M — interactive peakfinder tuning

Pick a backend from `src/hitfinders/`, tune its parameters, and re-run peak finding
over the first 20 Eiger4M frames without re-assembling anything.

**Why this exists.** The production config uses a single global `pf8_threshold: 800.0`
for every detector, but the detectors' backgrounds differ by ~190x:

| Detector | median non-hit pixel | % non-hit pixels > 800 | PF8 vs ground truth |
|---|---|---|---|
| AGIPD | 2667 | 93.5% | 50.0% (chance) |
| JUNGFRAU_4M | 14 | 1.6% | 81.7% |
| ePix10k | 57 | 7.6% | 91.7% |
| Eiger4M | 60 | 12.9% | 66.7% |

On the first 20 Eiger4M frames the stock parameters give 11/20 agreement with
`entry_1/labels/hit`, with 9 false positives and 0 false negatives. Goal here is to
find parameters that separate the classes — or to establish that they can't, because
the peak-count distributions overlap (non-hits at 42 and 64 peaks against true hits
at 4, 7 and 14).

**Pipeline order** matches production: assemble -> find_peaks on the *raw* assembled
frame, before GCN/LCN.

In [1]:
import os
import sys
import time

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# hdf5plugin MUST be imported before h5py to register the bitshuffle filter.
import hdf5plugin  # noqa: F401
import h5py

from reborn.detector import PADAssembler

from src.hitfinders import get_hitfinder
from src.preprocessing.geometry import get_geometry
from src.preprocessing.io import read_detector_description, read_frame
from src.preprocessing.pipeline import assemble_only

N_FRAMES = 20
CXI_PATH = "/data/bioxfel/user/gihan/Resonet/production/eiger4m_20k/compressed0.cxi"
LABEL_KEY = "entry_1/labels/hit"

print("project root:", PROJECT_ROOT)

2026-09-16 18:59:44,551:WARNING:reborn:__init__:<module>:

                     WARNING!!!
Failed to import the fortran-based modules in reborn.
Your code might still work since only a few parts of reborn
use fortran code, but you should set up your system so that 
numpy.f2py compiles fortran correctly. The most common issues are:
1) Your system does not have a fortran compiler. Solution: 
   Use a package manager to install gfortran or similar.  Any
   sensible AI prompt should be able to provide instructions.
2) Your system does not have the meson build system, which is 
   required for numpy versions greater than 1.26.  Solution:
   Use a package manager to install meson.  Any sensible AI 
   prompt should be able to provide instructions.
3) You upgraded numpy from version 1 to version 2, resulting in
   incompatible modules.  Solution: wipe out all of the existing
   compiled code with the command 
   $ python -m reborn --cleanup_fortran
4) You have an old numpy version (<1.26) and

project root: /data/bioxfel/user/gihan/Hit_finder


## 1. Assemble once

Assembly is the expensive step (~250 ms/frame on Eiger4M), so the frames are
assembled once and cached. Re-tuning parameters below only re-runs `find_peaks`.

The detector description is read **from the file** — never hardcoded — so the
geometry always matches the data.

In [2]:
desc = read_detector_description(CXI_PATH)
pads = get_geometry(desc)
assembler = PADAssembler(pad_geometry=pads)

with h5py.File(CXI_PATH, "r") as f:
    truth = f[LABEL_KEY][:N_FRAMES].astype(int).ravel()
    geom_meta = {
        "dist": float(f["entry_1/instrument_1/detector_1/distance"][()]),
        "wavelength": float(f["entry_1/instrument_1/source_1/wavelength"][()]),
        "pixel_size": float(f["entry_1/instrument_1/detector_1/x_pixel_size"][()]),
    }

assembled = []
t0 = time.perf_counter()
for i in range(N_FRAMES):
    raw = np.asarray(read_frame(CXI_PATH, i), dtype=np.float32)
    assembled.append(assemble_only(raw, pads, desc, assembler=assembler))

print(f"detector : {desc}  ({len(pads)} panels)")
print(f"assembled: {assembled[0].shape}   {N_FRAMES} frames in "
      f"{time.perf_counter() - t0:.1f}s")
print(f"geometry : {geom_meta}")
print(f"truth    : {truth.sum()} hits / {N_FRAMES} frames")

# Background scale for this detector -- the number the ADC threshold competes with.
nonhit = [assembled[i] for i in range(N_FRAMES) if truth[i] == 0]
if nonhit:
    v = np.concatenate([a.ravel() for a in nonhit])
    print(f"non-hit raw: median={np.median(v):.0f}  p99={np.percentile(v, 99):.0f}  "
          f"p99.9={np.percentile(v, 99.9):.0f}  max={v.max():.0f}")

detector : EIGER 4M  (64 panels)
assembled: (1687, 1687)   20 frames in 5.2s
geometry : {'dist': 0.3, 'wavelength': 1.4169622057142857e-10, 'pixel_size': 9.999925000562496e-05}
truth    : 8 hits / 20 frames
non-hit raw: median=23  p99=2231  p99.9=15419  max=65535


## 2. Controls

Pick a backend, set parameters, press **Run peakfinder**.

| Backend | Notes |
|---|---|
| `pf8` | CrystFEL PeakFinder8 via the C bridge — the production default |
| `pf8_numpy` | Pure NumPy reimplementation; slower, same parameters |
| `pf8_python` | ssc Cython wrapper; needs `ssc.peakfinder8_extension` |
| `gpu` | pyFAI `OCL_PeakFinder`; needs a GPU. Maps ADC->`MIN_INTENSITY`, min-SNR->`CUTOFF_PEAK`, min-pix->`CONNECTED`. Ignores max-pix. |
| `mock` | Fixed empty centroids — sanity check only |

`min-res` / `max-res` are distances from the frame centre in pixels; `0` disables the
cut. `min-res` is the useful one for killing beamstop and direct-beam artifacts.

In [3]:
_BOX = widgets.Layout(width="230px")
_STYLE = {"description_width": "130px"}

w_backend = widgets.Dropdown(
    options=["pf8", "pf8_numpy", "pf8_python", "gpu", "mock"],
    value="pf8", description="Peakfinder", layout=_BOX, style=_STYLE,
)
w_threshold = widgets.FloatText(value=800.0, description="ADC threshold", layout=_BOX, style=_STYLE)
w_min_snr = widgets.FloatText(value=5.0, description="min SNR", layout=_BOX, style=_STYLE)
w_min_pix = widgets.IntText(value=2, description="min pix (conn.)", layout=_BOX, style=_STYLE)
w_max_pix = widgets.IntText(value=200, description="max pix", layout=_BOX, style=_STYLE)
w_min_res = widgets.IntText(value=0, description="min res (px)", layout=_BOX, style=_STYLE)
w_max_res = widgets.IntText(value=0, description="max res (px)", layout=_BOX, style=_STYLE)
# Not in the requested six, but PF8 needs it and it changes the answer: the radius
# of the annulus used to estimate local background before the SNR test.
w_bg_radius = widgets.IntText(value=3, description="local bg radius", layout=_BOX, style=_STYLE)
# A frame counts as a hit when it has strictly more peaks than this. 0 reproduces
# the production rule (any peak at all => hit).
w_hit_thresh = widgets.IntText(value=0, description="hit if peaks >", layout=_BOX, style=_STYLE)

w_run = widgets.Button(description="Run peakfinder", button_style="primary",
                       layout=widgets.Layout(width="160px"))
w_status = widgets.HTML(value="<i>not run yet</i>")
w_out = widgets.Output()

# Populated by run(); the zoom viewer in section 3 reads them.
results = {"peaks": [None] * N_FRAMES, "n_peaks": np.zeros(N_FRAMES, int),
           "call": np.zeros(N_FRAMES, int), "params": {}}


def build_hitfinder():
    """Instantiate the selected backend from the widget values.

    Routed through get_hitfinder() with a synthetic cfg so the notebook builds
    the backend exactly the way training does.
    """
    cfg = {
        "hitfinder": {
            "backend": w_backend.value,
            "pf8_threshold": float(w_threshold.value),
            "pf8_min_snr": float(w_min_snr.value),
            "pf8_min_pix_count": int(w_min_pix.value),
            "pf8_max_pix_count": int(w_max_pix.value),
            "pf8_local_bg_radius": int(w_bg_radius.value),
            "pf8_min_res": int(w_min_res.value),
            "pf8_max_res": int(w_max_res.value),
            "pf8_use_saturated": False,
            "gpu_script_path": os.path.join(PROJECT_ROOT, "src", "hitfinders", "gpu_pf8.py"),
            "gpu_device": "cuda",
        }
    }
    hf = get_hitfinder(cfg)

    # The GPU backend has its own parameter names and needs beam geometry.
    if w_backend.value == "gpu":
        hf.set_geometry(**geom_meta)
        hf.set_params(
            MIN_INTENSITY=float(w_threshold.value),
            CUTOFF_PEAK=float(w_min_snr.value),
            CONNECTED=int(w_min_pix.value),
            MIN_RES=int(w_min_res.value),
            MAX_RES=int(w_max_res.value),
        )
    return hf, cfg["hitfinder"]


def run(_=None):
    w_status.value = "<i>running...</i>"
    try:
        hf, params = build_hitfinder()
    except Exception as exc:  # backend unavailable (no .so, no ssc, no GPU)
        w_status.value = f"<b style='color:crimson'>{type(exc).__name__}: {exc}</b>"
        return

    t0 = time.perf_counter()
    peaks, npk = [], np.zeros(N_FRAMES, int)
    for i in range(N_FRAMES):
        p = hf.find_peaks(assembled[i])
        peaks.append(np.asarray(p, dtype=np.float32).reshape(-1, 2))
        npk[i] = len(peaks[-1])
    elapsed = (time.perf_counter() - t0) / N_FRAMES * 1e3

    call = (npk > int(w_hit_thresh.value)).astype(int)
    results.update(peaks=peaks, n_peaks=npk, call=call, params=params)

    tp = int(((truth == 1) & (call == 1)).sum())
    fp = int(((truth == 0) & (call == 1)).sum())
    fn = int(((truth == 1) & (call == 0)).sum())
    tn = int(((truth == 0) & (call == 0)).sum())
    agree = tp + tn
    w_status.value = (
        f"<b>{w_backend.value}</b> &mdash; agreement <b>{agree}/{N_FRAMES}</b> "
        f"({100 * agree / N_FRAMES:.0f}%) &nbsp; TP={tp} FP={fp} FN={fn} TN={tn} "
        f"&nbsp; {elapsed:.0f} ms/frame"
    )

    with w_out:
        clear_output(wait=True)
        print(f"{'idx':>4} {'truth':>6} {'peaks':>6} {'call':>5}  verdict")
        print("-" * 44)
        for i in range(N_FRAMES):
            if truth[i] == call[i]:
                verdict = "agree"
            elif call[i] == 1:
                verdict = "FALSE POSITIVE"
            else:
                verdict = "FALSE NEGATIVE"
            print(f"{i:>4} {truth[i]:>6} {npk[i]:>6} {call[i]:>5}  {verdict}")
        for name, sel in (("truth=hit", truth == 1), ("truth=non-hit", truth == 0)):
            if sel.any():
                print(f"  peaks on {name:14s}: {np.sort(npk[sel])}")
        # Separability: if these ranges overlap, no peak-count cut can fix it.
        if (truth == 1).any() and (truth == 0).any():
            lo_hit, hi_non = npk[truth == 1].min(), npk[truth == 0].max()
            print(f"\n  min peaks on a hit = {lo_hit}, max peaks on a non-hit = {hi_non}"
                  f"  ->  {'SEPARABLE' if lo_hit > hi_non else 'OVERLAP'} by peak count")

        fig, axes = plt.subplots(4, 5, figsize=(21, 17))
        for i, ax in enumerate(axes.ravel()):
            img = assembled[i]
            vmin, vmax = np.percentile(img, [1, 99.5])
            ax.imshow(img, cmap="inferno", vmin=vmin, vmax=vmax, origin="upper")
            if len(results["peaks"][i]):
                pk = results["peaks"][i]
                ax.scatter(pk[:, 0], pk[:, 1], s=55, facecolors="none",
                           edgecolors="cyan", linewidths=0.8)
            ok = truth[i] == call[i]
            for spine in ax.spines.values():
                spine.set_edgecolor("limegreen" if ok else "orange")
                spine.set_linewidth(3)
            ax.set_title(f"#{i}  truth={truth[i]}  call={call[i]}  peaks={npk[i]}",
                         color="green" if ok else "darkorange", fontsize=11)
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle(
            f"{desc} frames 0-{N_FRAMES - 1} — {w_backend.value}  "
            f"(ADC={w_threshold.value:g}, SNR={w_min_snr.value:g}, "
            f"pix={w_min_pix.value}-{w_max_pix.value}, "
            f"res={w_min_res.value}-{w_max_res.value})",
            fontsize=14,
        )
        fig.tight_layout()
        plt.show()


w_run.on_click(run)

display(widgets.VBox([
    widgets.HBox([w_backend, w_threshold, w_min_snr]),
    widgets.HBox([w_min_pix, w_max_pix, w_bg_radius]),
    widgets.HBox([w_min_res, w_max_res, w_hit_thresh]),
    widgets.HBox([w_run, w_status]),
    w_out,
]))
run()

## 3. Single-frame zoom

Step through frames to see where the peaks actually land. Re-run section 2 after
changing parameters, then use this to check whether surviving peaks sit on Bragg
spots or on panel edges, the beamstop, and other detector artifacts.

In [4]:
z_slider = widgets.IntSlider(value=0, min=0, max=N_FRAMES - 1, description="Frame",
                             continuous_update=False,
                             layout=widgets.Layout(width="460px"))
z_back = widgets.Button(description="< Back", layout=widgets.Layout(width="90px"))
z_fwd = widgets.Button(description="Forward >", layout=widgets.Layout(width="100px"))
z_out = widgets.Output()


def z_render(i: int) -> None:
    with z_out:
        clear_output(wait=True)
        img = assembled[i]
        pk = results["peaks"][i]
        pk = pk if pk is not None else np.zeros((0, 2), np.float32)
        ok = truth[i] == results["call"][i]

        fig, ax = plt.subplots(figsize=(9, 9))
        vmin, vmax = np.percentile(img, [1, 99.5])
        ax.imshow(img, cmap="inferno", vmin=vmin, vmax=vmax, origin="upper")
        if len(pk):
            ax.scatter(pk[:, 0], pk[:, 1], s=90, facecolors="none",
                       edgecolors="cyan", linewidths=1.2, label=f"{len(pk)} peaks")
            ax.legend(loc="upper right", fontsize=9)
        for spine in ax.spines.values():
            spine.set_edgecolor("limegreen" if ok else "orange")
            spine.set_linewidth(4)
        ax.set_title(
            f"Frame {i}  |  truth={'HIT' if truth[i] else 'NON-HIT'}  |  "
            f"call={'HIT' if results['call'][i] else 'NON-HIT'}  |  "
            f"{len(pk)} peaks  |  {'correct' if ok else 'WRONG'}",
            fontsize=12,
        )
        ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
        fig.tight_layout()
        plt.show()


z_slider.observe(lambda c: z_render(c["new"]), names="value")
z_back.on_click(lambda _: setattr(z_slider, "value", max(0, z_slider.value - 1)))
z_fwd.on_click(lambda _: setattr(z_slider, "value", min(N_FRAMES - 1, z_slider.value + 1)))

display(widgets.VBox([widgets.HBox([z_back, z_fwd, z_slider]), z_out]))
z_render(0)